# Contar árboles en Los Rosales (Bogotá) con modelos del Hugging Face Hub

Cuaderno del artículo [Hugging Face](https://mromerot.github.io/web/datascience/hugging-face.html) de [mromerot.github.io/web](https://mromerot.github.io/web/). Estima el arbolado de un barrio a partir de una imagen aérea con dos modelos del Hub y lo compara con el censo de arbolado urbano usado por [SIDRo](https://mromerot.github.io/SIDRo/).

- **DeepForest** ([`weecology/deepforest-tree`](https://huggingface.co/weecology/deepforest-tree), MIT): detecta copas individuales.
- **TCD SegFormer** ([`restor/tcd-segformer-mit-b5`](https://huggingface.co/restor/tcd-segformer-mit-b5)): segmenta la cobertura arbórea.

> **Alcance.** Es una demostración del flujo, no una medición validada. Las teselas son de Esri World Imagery (~28 cm/px en Bogotá, ampliadas a ~10 cm por interpolación); para un producto publicado use una ortofoto con licencia abierta o un vuelo de dron.

Active la GPU en *Entorno de ejecución → Cambiar tipo de entorno*.

## 1. Entorno

In [ ]:
%pip install -q deepforest transformers mercantile rasterio geopandas pyarrow

## 2. Descargar la imagen del barrio

In [ ]:
import io, requests, mercantile, numpy as np, rasterio
from PIL import Image
from rasterio.transform import from_bounds

# Subcuadro de unas 7 ha de Los Rosales: corre en pocos minutos.
# Para el barrio completo (config/aoi.yaml -> geometria.bbox_respaldo de SIDRo) use:
# BBOX = [-74.0553, 4.6497, -74.0448, 4.6647]
BBOX = [-74.0520, 4.6560, -74.0500, 4.6575]
ZOOM, ESCALA = 19, 3    # zoom 19 en Bogotá ~0,28 m/px; se amplía x3 (~0,10 m/px) para acercarse a la resolución de entrenamiento
URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"

tiles = list(mercantile.tiles(*BBOX, ZOOM))
xs = sorted({t.x for t in tiles}); ys = sorted({t.y for t in tiles})
mosaico = Image.new("RGB", (256 * len(xs), 256 * len(ys)))
for t in tiles:
    r = requests.get(URL.format(z=ZOOM, x=t.x, y=t.y), timeout=30, headers={"User-Agent": "sidro-ejemplo"})
    r.raise_for_status()
    mosaico.paste(Image.open(io.BytesIO(r.content)).convert("RGB"), ((t.x - xs[0]) * 256, (t.y - ys[0]) * 256))

oeste = mercantile.xy_bounds(xs[0], ys[0], ZOOM); este = mercantile.xy_bounds(xs[-1], ys[-1], ZOOM)
mosaico = mosaico.resize((mosaico.width * ESCALA, mosaico.height * ESCALA), Image.BICUBIC)
arr = np.asarray(mosaico).transpose(2, 0, 1)
transformacion = from_bounds(oeste.left, este.bottom, este.right, oeste.top, arr.shape[2], arr.shape[1])
with rasterio.open("ortofoto.tif", "w", driver="GTiff", height=arr.shape[1], width=arr.shape[2], count=3,
                   dtype="uint8", crs="EPSG:3857", transform=transformacion) as dst:
    dst.write(arr)
print(f"{len(tiles)} teselas, imagen de {arr.shape[2]} x {arr.shape[1]} px, {(este.right - oeste.left) / arr.shape[2]:.2f} m/px")

## 3. Detectar copas con DeepForest

Cada copa viene con una puntuación de confianza; cuántas cuentan como árbol depende del umbral.

In [ ]:
import geopandas as gpd
from deepforest import main

modelo = main.deepforest()
modelo.load_model("weecology/deepforest-tree")      # modelo preentrenado del Hub (licencia MIT)
cajas = modelo.predict_tile(path="ortofoto.tif", patch_size=400, patch_overlap=0.25, iou_threshold=0.15)

# Las cajas vienen en píxeles: se convierten al centroide de cada copa en coordenadas del mapa
with rasterio.open("ortofoto.tif") as src:
    t = src.transform
cx = (cajas.xmin + cajas.xmax) / 2; cy = (cajas.ymin + cajas.ymax) / 2
copas = gpd.GeoDataFrame(
    cajas[["score"]].assign(ancho_m=(cajas.xmax - cajas.xmin) * t.a, alto_m=(cajas.ymax - cajas.ymin) * -t.e),
    geometry=gpd.points_from_xy(t.c + cx * t.a, t.f + cy * t.e), crs="EPSG:3857").to_crs("EPSG:4326")
print(f"{len(copas):,} candidatos en total")

In [ ]:
conteo_umbral = {u: int((copas.score >= u).sum()) for u in (0.1, 0.2, 0.3, 0.4)}
# La puntuación decide qué es un árbol: el conteo cambia mucho con el umbral
for u, n in conteo_umbral.items():
    print(f"puntuación >= {u}: {n:,} copas")

## 4. Elegir el umbral y exportar

Salidas en EPSG:4326 (visores) y EPSG:9377 (CRS de trabajo de SIDRo).

In [ ]:
UMBRAL = 0.2      # valor ilustrativo; con el modelo preentrenado el conteo cambia mucho con este umbral
detectados = copas[copas.score >= UMBRAL].copy()

detectados[["score", "geometry"]].to_file("arboles_detectados.geojson", driver="GeoJSON")
detectados.to_crs("EPSG:9377").to_parquet("arboles_detectados_9377.parquet")   # CRS de trabajo de SIDRo
print(f"Árboles detectados: {len(detectados):,}")

## 5. Cobertura arbórea con TCD SegFormer

No cuenta individuos, pero describe la copa continua que DeepForest separa mal cuando varias copas se tocan. Es sensible a las sombras: revise el resultado sobre la imagen.

In [ ]:
import torch
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation

nombre = "restor/tcd-segformer-mit-b5"
proc = AutoImageProcessor.from_pretrained(nombre)
seg = SegformerForSemanticSegmentation.from_pretrained(nombre).eval()

arbol = total = 0
with rasterio.open("ortofoto.tif") as src:
    for y in range(0, src.height, 1024):
        for x in range(0, src.width, 1024):
            ventana = rasterio.windows.Window(x, y, min(1024, src.width - x), min(1024, src.height - y))
            parche = Image.fromarray(src.read(window=ventana).transpose(1, 2, 0))
            with torch.no_grad():
                logits = seg(**proc(images=parche, return_tensors="pt")).logits
            mascara = torch.nn.functional.interpolate(logits, size=parche.size[::-1], mode="bilinear").argmax(1)[0]
            arbol += int((mascara == 1).sum()); total += mascara.numel()
print(f"Cobertura arbórea estimada: {arbol / total:.1%} del rectángulo")

## 6. Comparar con el censo de SIDRo

In [ ]:
import pandas as pd

# Censo de Arbolado Urbano (SIGAU, Jardín Botánico de Bogotá; datos abiertos, CC BY 4.0), recortado al polígono de SIDRo
URL_CENSO = "https://raw.githubusercontent.com/mromerot/web/main/datascience/_colab/censo_arbolado_rosales.csv"
censo = pd.read_csv(URL_CENSO)
censo = gpd.GeoDataFrame(censo, geometry=gpd.points_from_xy(censo.lon, censo.lat), crs="EPSG:4326")

# Se compara solo dentro de la imagen descargada
with rasterio.open("ortofoto.tif") as src:
    izq, abajo, der, arriba = src.bounds
censo_3857 = censo.to_crs("EPSG:3857").cx[izq:der, abajo:arriba]
det_3857 = detectados.to_crs("EPSG:3857")
print(f"En la imagen: {len(det_3857):,} copas detectadas | {len(censo_3857):,} árboles en el censo SIGAU "
      f"| razón detectados/censo: {len(det_3857) / len(censo_3857):.2f}")

## 7. Mapa: detecciones frente al censo

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open("ortofoto.tif") as src:
    paso = max(1, src.width // 1800)
    img = src.read(out_shape=(3, src.height // paso, src.width // paso)).transpose(1, 2, 0)
ext = (izq, der, abajo, arriba)

fig, ax = plt.subplots(figsize=(9, 9 * (arriba - abajo) / (der - izq)))
ax.imshow(img, extent=ext)
ax.scatter(det_3857.geometry.x, det_3857.geometry.y, s=26, facecolors="none", edgecolors="#f28e2b", linewidths=0.9,
           label=f"Copas detectadas por DeepForest (puntuación ≥ {UMBRAL}): {len(det_3857):,}")
ax.scatter(censo_3857.geometry.x, censo_3857.geometry.y, s=16, c="#2a78d6", edgecolors="white", linewidths=0.6,
           label=f"Árboles del censo SIGAU: {len(censo_3857):,}")
ax.set_axis_off()
ax.legend(loc="lower left", fontsize=9, framealpha=0.92)
ax.set_title("Arbolado de Los Rosales: detección con un modelo del Hub frente al censo", fontsize=11, loc="left")
fig.text(0.01, 0.01, "Imagen: Esri World Imagery. Censo: SIGAU, Jardín Botánico de Bogotá (CC BY 4.0).", fontsize=7, color="#555")
fig.savefig("mapa_arboles.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Conteo según el umbral

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
umbrales = [str(u).replace(".", ",") for u in conteo_umbral]
barras = ax.bar(umbrales, list(conteo_umbral.values()), color="#2a78d6", width=0.55)
ax.bar_label(barras, labels=[f"{n:,}".replace(",", ".") for n in conteo_umbral.values()], padding=3, fontsize=9)
ax.axhline(len(censo_3857), color="#3a3a3a", linestyle="--", linewidth=1.2)
ax.text(len(umbrales) - 0.5, len(censo_3857), f"  Censo SIGAU: {len(censo_3857):,}".replace(",", "."), va="bottom", ha="right", fontsize=9, color="#3a3a3a")
ax.set_xlabel("Puntuación mínima de confianza para contar una copa como árbol")
ax.set_ylabel("Árboles contados")
ax.set_title("El conteo cambia por un factor de decenas según el umbral", fontsize=11, loc="left")
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8); ax.set_axisbelow(True)
fig.savefig("barras_umbral.png", dpi=150, bbox_inches="tight")
plt.show()

## Qué sigue

1. **Ajuste fino (*fine-tuning*)** de DeepForest con copas etiquetadas a mano en Los Rosales; el censo sirve para validar, no para calibrar el umbral.
2. Cruzar `arboles_detectados_9377.parquet` con las manzanas de SIDRo para obtener árboles por hectárea (`arboles_ha`, variable del análisis de calor 3.5).
3. Tener presente que el censo SIGAU registra sobre todo árboles de espacio público (calles, parques, zonas verdes) y no los de predios privados, y que ~63 % de sus registros son de 2005 a 2009. Por eso, aunque un modelo contara perfecto, **no tendría por qué coincidir con el censo**: la imagen muestra el dosel actual de todo el barrio.